# Iris データセットの探索と分類モデル

このノートブックでは、Iris データセットを使って以下を行います：

1. データの読み込みと確認
2. データの統計情報の確認
3. データの可視化
4. 分類モデルの訓練と評価

## 1. ライブラリのインポート

In [2]:
import { DataFrame } from 'data-forge';
import * as fs from 'fs';
import { IrisClassifier } from '../src/models/IrisClassifier';

console.log('✓ ライブラリが読み込まれました');

3:32 - File 'C:/Users/PC202411-1/IdeaProjects/case-study-game-dev/app/typescript/src/models/IrisClassifier.ts' is not under 'rootDir' 'C:\Users\PC202411-1\IdeaProjects\case-study-game-dev\app\typescript\notebooks'. 'rootDir' is expected to contain all source files.


## 2. データの読み込み

In [3]:
// CSV ファイルを読み込む
const csvContent = fs.readFileSync('../data/iris.csv', 'utf-8');

// CRLF と LF の両方に対応してパース
const lines = csvContent.trim().split(/\r?\n/);
const headers = lines[0].split(',').map((h) => h.trim());

const rows = lines.slice(1).filter(line => line.trim()).map((line) => {
  const values = line.split(',').map((v) => v.trim());
  const obj: any = {};
  headers.forEach((header, index) => {
    obj[header] = values[index];
  });
  return obj;
});

const df = new DataFrame(rows);

console.log(`データ行数: ${df.count()}`);
console.log(`列名: ${df.getColumnNames().join(', ')}`);

2:20 - Cannot find name 'fs'.
17:16 - Cannot find name 'DataFrame'.


## 3. データの確認

In [4]:
// 最初の 5 行を表示
const firstFive = df.head(5).toArray();
console.table(firstFive);

2:19 - Cannot find name 'df'.


## 4. 統計情報の確認

In [5]:
// がく片の長さの統計情報
const sepalLengthSeries = df.getSeries('sepal_length').select(v => Number(v));

const stats = {
  count: sepalLengthSeries.count(),
  mean: sepalLengthSeries.average().toFixed(2),
  min: sepalLengthSeries.min().toFixed(2),
  max: sepalLengthSeries.max().toFixed(2),
  std: sepalLengthSeries.std().toFixed(2),
};

console.log('【がく片の長さ (sepal_length) の統計情報】');
console.table(stats);

2:27 - Cannot find name 'df'.


## 5. 種類ごとのデータ件数

In [ ]:
// 種類ごとにカウント
const speciesCounts = df
  .groupBy((row) => row.species)
  .select((group) => ({
    species: group.first().species,
    count: group.count(),
  }))
  .toArray();

console.log('【種類ごとのデータ件数】');
console.table(speciesCounts);

## 6. データのフィルタリング

In [ ]:
// setosa のデータだけを抽出
const setosaData = df.where((row) => row.species === 'setosa');

console.log(`setosa の件数: ${setosaData.count()}`);
console.table(setosaData.head(3).toArray());

## 7. 分類モデルの訓練

In [ ]:
// 分類器の作成
const classifier = new IrisClassifier();

// データの読み込み
await classifier.loadData('../data/iris.csv');
console.log(`✓ データを読み込みました: ${classifier.getDataSize()} サンプル`);

// データの分割
classifier.splitData(0.2);
console.log(
  `✓ データを分割: 訓練 ${classifier.getTrainSize()}, テスト ${classifier.getTestSize()}`
);

// 訓練
await classifier.train(5);
console.log('✓ モデルの訓練が完了しました');

## 8. モデルの評価

In [ ]:
const accuracy = classifier.evaluate();
console.log(`正解率（Accuracy）: ${(accuracy * 100).toFixed(2)}%`);

## 9. 予測の実行

In [ ]:
// 予測サンプル
const samples = [
  { features: [5.1, 3.5, 1.4, 0.2], expected: 'setosa' },
  { features: [7.0, 3.2, 4.7, 1.4], expected: 'versicolor' },
  { features: [6.3, 3.3, 6.0, 2.5], expected: 'virginica' },
];

console.log('【予測例】');
samples.forEach((sample, i) => {
  const prediction = classifier.predictOne(sample.features);
  const result = prediction === sample.expected ? '✓' : '✗';

  console.log(`\nサンプル ${i + 1}: ${result}`);
  console.log(
    `  特徴量: [${sample.features.map((f) => f.toFixed(1)).join(', ')}]`
  );
  console.log(`  予測: ${prediction}`);
  console.log(`  期待: ${sample.expected}`);
});

## まとめ

このノートブックでは以下を行いました：

1. Iris データセットの読み込みと確認
2. 統計情報の計算
3. データのフィルタリング
4. 決定木による分類モデルの訓練
5. モデルの評価と予測

高い精度で Iris の種類を分類できることが確認できました！